# 03 · Segmentación exploratoria de clientes

Ruta de continuación con las seis fases de CRISP-DM. Usaremos agregados de recencia, frecuencia, gasto en precios de artículos y variedad de categorías para explorar patrones con K-Means. Los grupos son una herramienta exploratoria, no categorías verdaderas ni identidades de clientes.

## 1. Comprensión del negocio

La pregunta es si aparecen patrones de compra similares que merezcan investigación adicional. El análisis no asigna perfiles reales ni recomienda tratamientos distintos. En particular, `customer_unique_id` solo agrupa transacciones y no se utiliza como variable.

## 2. Comprensión de los datos

Se necesitan clientes, pedidos, líneas de pedido y productos. Olist permite agrupar pedidos con `customer_unique_id`, asociar fecha/estado del pedido, precio de línea y categoría de producto. No hay inventario ni cantidad explícita por línea. Si faltan Parquet, descargue Olist y ejecute `python scripts/prepare_data.py`; revise el [diccionario](docs/diccionario_datos.md).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("data/processed")
required_files = [
    "olist_orders_dataset.parquet",
    "olist_customers_dataset.parquet",
    "olist_order_items_dataset.parquet",
    "olist_products_dataset.parquet",
]
missing = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"Faltan {missing} en {DATA_DIR}. Descargue Olist, coloque los CSV en "
        "data/raw/ y ejecute `python scripts/prepare_data.py` desde la raíz."
    )
orders = pd.read_parquet(DATA_DIR / required_files[0])
customers = pd.read_parquet(DATA_DIR / required_files[1])
items = pd.read_parquet(DATA_DIR / required_files[2])
products = pd.read_parquet(DATA_DIR / required_files[3])
print(f"Pedidos: {len(orders):,}; líneas de artículo: {len(items):,}")

## 3. Preparación de los datos

Filtramos pedidos entregados con fecha válida y asociamos los identificadores seudónimos. El gasto es la suma de `price` por línea (no incluye flete). La variedad cuenta categorías distintas; categorías ausentes se excluyen de ese conteo. La recencia se calcula frente al día posterior a la última compra observada.

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"], errors="coerce"
)
orders = orders.merge(
    customers[["customer_id", "customer_unique_id"]], on="customer_id", how="inner"
)
delivered = orders.loc[
    orders["order_status"].eq("delivered")
    & orders["order_purchase_timestamp"].notna()
    & orders["customer_unique_id"].notna()
].copy()
if delivered.empty:
    raise ValueError("No hay pedidos entregados con cliente y fecha válida.")
reference_date = delivered["order_purchase_timestamp"].max().normalize() + pd.Timedelta(days=1)
order_map = delivered[["order_id", "customer_unique_id", "order_purchase_timestamp"]]
purchases = items.merge(order_map, on="order_id", how="inner").merge(
    products[["product_id", "product_category_name"]], on="product_id", how="left"
)
customer_features = purchases.groupby("customer_unique_id").agg(
    frequency=("order_id", "nunique"),
    spend_brl=("price", "sum"),
    category_variety=("product_category_name", "nunique"),
    last_purchase=("order_purchase_timestamp", "max"),
)
customer_features["recency_days"] = (
    reference_date - customer_features.pop("last_purchase")
).dt.total_seconds() / 86400
feature_columns = ["recency_days", "frequency", "spend_brl", "category_variety"]
customer_features = customer_features[feature_columns].dropna()
print(f"Clientes agregados: {len(customer_features):,}")
display(customer_features.describe().round(1))

## 4. Modelado

K-Means usa distancias, por lo que escalamos las variables antes de agrupar. Cuatro grupos son un punto de partida didáctico, no un número descubierto como verdadero. El ajuste es reproducible con semilla fija; conviene explorar otros valores de K y revisar estabilidad en un análisis posterior.

In [ ]:
n_clusters = 4
if len(customer_features) < n_clusters:
    raise ValueError(f"K-Means con {n_clusters} grupos requiere al menos ese número de clientes.")
cluster_model = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=n_clusters, random_state=42, n_init=10),
)
customer_features["cluster"] = cluster_model.fit_predict(customer_features[feature_columns])
cluster_profile = customer_features.groupby("cluster")[feature_columns].mean().round(1)
display(cluster_profile)

## 5. Evaluación

Proyectamos las características escaladas a dos componentes principales para visualizar relaciones; el gráfico no conserva toda la información. Comparamos promedios en unidades originales para describir los grupos, sin llamarlos segmentos verdaderos. El modelo no conoce etiquetas de negocio ni demuestra causalidad.

In [ ]:
scaled_features = cluster_model.named_steps["standardscaler"].transform(
    customer_features[feature_columns]
)
projection = PCA(n_components=2, random_state=42).fit_transform(scaled_features)
plt.figure(figsize=(8, 5))
scatter = plt.scatter(
    projection[:, 0], projection[:, 1],
    c=customer_features["cluster"], cmap="tab10", alpha=0.65, s=18,
)
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.title("K-Means · proyección PCA exploratoria")
plt.colorbar(scatter, label="Grupo asignado")
plt.show()
print("Perfiles promedio en las unidades originales (no son etiquetas reales):")
display(cluster_profile)

## 6. Despliegue (demostración temporal)

Ingrese características agregadas de ejemplo para ver el grupo más cercano. Esto no clasifica a una persona real ni crea categorías verdaderas. La interfaz de Colab es temporal y desaparece al cerrar la sesión; no es un despliegue de producción.

In [ ]:
import gradio as gr

def cluster_demo(recency_days, frequency, spend_brl, category_variety):
    example = pd.DataFrame(
        [[recency_days, frequency, spend_brl, category_variety]],
        columns=feature_columns,
    )
    group = int(cluster_model.predict(example)[0])
    return f"Grupo exploratorio asignado: {group}. No es una categoría verdadera ni una decisión sobre una persona."

demo = gr.Interface(
    fn=cluster_demo,
    inputs=[
        gr.Number(value=60, minimum=0, label="Recencia (días)"),
        gr.Number(value=2, minimum=1, label="Pedidos observados"),
        gr.Number(value=200, minimum=0, label="Suma de precios (BRL)"),
        gr.Number(value=2, minimum=0, label="Categorías distintas"),
    ],
    outputs=gr.Textbox(label="Resultado exploratorio"),
    title="Demo de K-Means",
    description="Pruebe valores agregados; los grupos dependen de los datos y del escalado.",
)
demo.launch(share=False, inline=True, prevent_thread_lock=True)